---
title: "Which Firmware Does This Thing Take? Fingerprinting an R36S Clone"
author: "Vahram Poghosyan"
date: "2026-09-13"
categories: ["Gaming", "Retro Gaming", "Linux", "Device Trees", "Reverse Engineering"]
format:
  html:
    code-fold: true
jupyter: python3
include-after-body:
  text: |
    <script type="application/javascript" src="../../javascript/light-dark.js"></script>
---

# The problem with cheap handhelds

I bought an R36S. Or something like one. That hedge is the entire subject of this post.

The R36S is an RK3326-based retro handheld that is *itself* a clone of Anbernic's design, and it has in turn been cloned by a dozen anonymous factories. The result is a family of devices that all look identical, all say "R36S" on the box, and differ internally in exactly the way that matters: **the screen panel**. Different panels need different initialization commands. Send the wrong ones and you get a black screen, an inverted screen, or a beautiful display of static.

So when I went to install [dArkOS](https://github.com/christianhaitian/dArkOS) — a Debian-based custom firmware for these things — I was staring at a download list like this:

```
dArkOS_A10MINI_trixie_09022026.img.7z.001
dArkOS_G350_trixie_09022026.img.7z.001
dArkOS_MINILOONG_trixie_09042026.img.7z.001
dArkOS_RG351MP_trixie_09022026.img.7z.001
dArkOS_RG353m_trixie_08282026.img.7z.001
```

None of them say "R36S." The obvious move is to guess the closest-looking one, flash it, and see what happens. I'd like to describe a better move.

## Step 0: the listed images were all wrong

Those filenames are device codenames, and they map to completely unrelated hardware:

| Image | Actual device |
|---|---|
| `A10MINI` | A10 Mini |
| `G350` | BatleXP G350 |
| `MINILOONG` | Miniloong |
| `RG351MP` | Anbernic RG351MP |
| `RG353m` | Anbernic RG353M |

The R36 family lives in a **separate fork**: [southoz/dArkOSRE-R36](https://github.com/southoz/dArkOSRE-R36), which explicitly covers R36S variants *and* clone boards. One image, many device trees.

That reframes the question. It's no longer "which image do I download" — there's only one. It's **"which device tree blob (`.dtb`) do I put on the boot partition after flashing."** That repo ships 44 of them.

## Step 1: getting macOS to read the stock card

The authoritative answer is already on the device: whatever `.dtb` the stock firmware boots with is, by definition, the one that works on this exact panel.

Plug the stock SD card into a Mac and you get:

> *The disk is not readable by this computer.*

Click **Ignore**, never Eject. macOS is complaining about the ext4 partitions it can't mount; that says nothing about the FAT boot partition. Look at what's actually there:

```bash
diskutil list
```

```
/dev/disk4 (external, physical):
   #:                       TYPE NAME            SIZE       IDENTIFIER
   0:     FDisk_partition_scheme                *15.6 GB    disk4
   1:          DOS_FAT_16_Hidden                 1.1 GB     disk4s1
   2:                      Linux                 536.9 MB   disk4s2
   3:                      Linux                 3.0 GB     disk4s3
   4:                       0x1C                 11.0 GB    disk4s4
```

There it is. `disk4s1` is FAT — but its partition type byte is `0x16` (*hidden* FAT16), and `disk4s4` is `0x1C` (*hidden* FAT32). macOS refuses to auto-mount hidden types, which is the whole reason for the scary dialog. `diskutil mount` also declines ("no file system"), so mount it by hand, read-only:

```bash
sudo mkdir -p /Volumes/r36boot
sudo mount -t msdos -o rdonly /dev/disk4s1 /Volumes/r36boot
```

`-o rdonly` is not optional paranoia. This card is the only copy of a working configuration that exists anywhere.

## Step 2: reading the boot chain

```
battery_0.bmp ... battery_fail.bmp
boot.ini
extlinux/
KERNEL          (30 MB)
SYSTEM          (924 MB)
rf351g3ka.dtb   <-- this
logo.bmp
```

`boot.ini` turns out to be scavenged from an ODROID-GO Advance:

```bash
ODROIDGOA-UBOOT-CONFIG
...
if test ${hwrev} = 'v11'; then
sysboot mmc 1:1 any ${scriptaddr} /extlinux/rk3326-odroidgo2-linux-v11.dtb.conf
elif test ${hwrev} = 'v10-go3'; then
...
```

Note that every `.conf` it names **does not exist on the card**. U-Boot falls through to the standard `extlinux/extlinux.conf`, which is the file actually in charge:

```
LABEL EmuELEC
  LINUX /KERNEL
  FDT /rf351g3ka.dtb
  APPEND ... fbcon=rotate:3 ...
```

An ODROID U-Boot config, an EmuELEC label, and a device tree named `rf351g3ka`. This firmware is a collage. That's normal for these devices and it's a hint about what we're dealing with.

## Step 3: decoding a DTB without `dtc`

`rf351g3ka.dtb` is a Flattened Device Tree — a binary description of every piece of hardware on the board. `strings` gets you tantalizingly close:

```
rockchip,rk3326-evb-lp3-v12-linux
sitronix,st7703
simple-panel-dsi
```

but not the numbers. The proper tool is `dtc`, which macOS doesn't ship and which I didn't feel like installing. The FDT format is simple enough to parse directly — a header, a string table, and a token stream of `BEGIN_NODE` / `PROP` / `END_NODE`:

```python
import struct

def parse(data):
    magic, total, off_struct, off_strings, off_rsv, ver, lc, bc, size_str, size_struct = \
        struct.unpack('>10I', data[:40])
    assert magic == 0xd00dfeed
    strings = data[off_strings:off_strings + size_str]

    def cstr(buf, off):
        return buf[off:buf.index(b'\0', off)].decode('utf-8', 'replace')

    p, end, path, out = off_struct, off_struct + size_struct, [], {}
    while p < end:
        tok = struct.unpack('>I', data[p:p+4])[0]; p += 4
        if tok == 1:                                  # FDT_BEGIN_NODE
            name = cstr(data, p); p += (len(name) + 4) & ~3
            path.append(name)
        elif tok == 2:                                # FDT_END_NODE
            path.pop()
        elif tok == 3:                                # FDT_PROP
            plen, noff = struct.unpack('>II', data[p:p+8]); p += 8
            val = data[p:p+plen]; p += (plen + 3) & ~3
            node = '/' + '/'.join(x for x in path if x)
            out.setdefault(node, {})[cstr(strings, noff)] = val
        elif tok == 4:                                # FDT_NOP
            continue
        elif tok == 9:                                # FDT_END
            break
    return out
```

Forty lines, no dependencies. Pointed at the panel node, it gives up the real story:

```
/dsi@ff450000/panel@0 : compatible = "sitronix,st7703", "simple-panel-dsi"
/dsi@ff450000/panel@0 : dsi,lanes = <4>
/dsi@ff450000/panel@0 : width-mm = <153>
/dsi@ff450000/panel@0 : height-mm = <85>
.../display-timings/timing0 : clock-frequency = <30000000>
.../display-timings/timing0 : hactive = <640>
.../display-timings/timing0 : vactive = <480>
.../display-timings/timing0 : hfront-porch = <46>
.../display-timings/timing0 : hback-porch  = <44>
```

Three things fall out of this:

1. **The board never got rebranded.** `rockchip,rk3326-evb-lp3-v12-linux` is Rockchip's *generic reference board* string. A real R36S identifies as `rk3326-r35s-linux` or `gameconsole-r36s`. The community ["Which Screen Do I Have?"](https://aeolusux.github.io/ArkOS-R3XS/tools/dtbIdentify.htm) tool only accepts those two filenames — it can't read mine, which is itself the diagnosis.
2. **The panel is MIPI-DSI**, driven by a Sitronix ST7703, not a parallel-RGB panel. Any device tree that drives RGB will produce a black screen here no matter how correct the rest of the board is.
3. **`width-mm = 153, height-mm = 85`** describes a 16:9 panel about 6.9" across — nonsense for a 3.5" handheld. Those values were copy-pasted from a reference design and never corrected. Confirmation, if any were needed, that nobody authored this DTS for this hardware.

## Step 4: the fingerprint

Here's the part I actually think is worth stealing.

A DSI panel is brought up by squirting a sequence of vendor-specific commands at the controller. In the device tree that's a single property, `panel-init-sequence`, and it is an opaque blob — gamma curves, power settings, timing quirks, tuned per panel model. Mine:

```
panel-init-sequence: 753 bytes, sha256[:16] = ee05de3b4a3a54ca
```

**That blob is a fingerprint.** Two device trees with the same init sequence are talking to the same physical panel, whatever the surrounding file claims to be. It's far more reliable than the `compatible` string, which on `simple-panel-dsi` devices is close to decorative.

So: pull down all 44 `.dtb` files from the R36 repo and hash every one of their init sequences.

```python
targets = [t['path'] for t in tree['tree'] if t['path'].endswith('rk3326-r36s-linux.dtb')]
# ... curl them all, parse, hash panel-init-sequence, compare to mine
```

(A note for anyone following along: Python's `urllib` on a stock macOS has no CA bundle and every HTTPS fetch dies with `CERTIFICATE_VERIFY_FAILED`. `curl --parallel -K urls.txt` sidesteps it entirely.)

Results — **6 hits out of 44**, and every single one sits in the repo's `clone/` directory rather than `r36s/` or `soysauce/`:

| Entry | board `compatible` | panel label | status |
|---|---|---|---|
| **R36S-V12 2023-08-18 Variant 3 Panel 3** | `rk3326-r36s-linux` | **`sitronix,st7703`** | completed |
| R36S-V20 2025-05-18 2541 | `rk3326-r36s-linux` | `simple-panel-dsi` | beta |
| R36S-V12 2023-08-18 Variant 1 Panel 3 A | `rk3326-g80ca-linux` | `elida,kd35t133` | untested |
| G80C-MB V1.1-20250319 Panel 9 | `rk3326-odroidgo3-linux` | `elida,kd35t133` | untested |
| G80CA-MB V1.2-20250422 Panel 9 | `rk3326-odroidgo3-linux` | `elida,kd35t133` | untested |
| G80CA-MB V1.2-20250423 Panel 9 | `rk3326-odroidgo3-linux` | `elida,kd35t133` | untested |

Look at the middle column. The *same physical panel* appears under three different `compatible` strings — `sitronix,st7703`, `elida,kd35t133`, and plain `simple-panel-dsi`. If I had matched on the label, as the obvious approach would, I'd have been wrong five times out of six. The blob doesn't lie.

Of the 38 rejects: 37 use a different init sequence under `elida,kd35t133`, and the few other ST7703 entries carry blobs `3f7a1a32…` or `8ea86fcc…`. Not mine.

**Best candidate: `R36S-V12 2023-08-18 Variant 3 Panel 3`** — the only one matching on *both* the blob and the `compatible` string, and one of the community-tested entries rather than an untested guess.

## Step 5: unpacking a `.7z` with no Homebrew

macOS can't open 7-Zip archives in Finder and `unzip` won't touch them. You don't need a package, though — the bundled `bsdtar` is libarchive, and libarchive reads 7z:

```bash
bsdtar -tvf dArkOSRE_R36_trixie_03082026.7z     # list
bsdtar -xvf dArkOSRE_R36_trixie_03082026.7z     # extract
```

3.7 GB → an 8.3 GB `.img` in 1m50s. (For the *split* `.7z.001` / `.002` archives that the main dArkOS repo publishes, `bsdtar` gets unreliable — `brew install sevenzip` and point `7zz` at the `.001`.)

## Step 6: the image would have failed out of the box

Before flashing, I mounted the extracted image read-only to check what device tree it ships by default:

```bash
hdiutil attach -nomount -readonly -imagekey diskimage-class=CRawDiskImage dArkOSRE_R36_trixie_03082026.img
diskutil mount readOnly /dev/disk5s1
```

And ran the same fingerprint over it:

| | board | panel | init blob |
|---|---|---|---|
| My stock card | `rk3326-evb-lp3-v12` | `sitronix,st7703` | `ee05de3b…` (753 B) |
| **Image default** | `rk3326-odroidgo3` | `elida,kd35t133` | `3f7a1a32…` (370 B) |
| `clone/…Variant 3 Panel 3/` | `rk3326-r36s` | `sitronix,st7703` | **`ee05de3b…` (753 B)** |

**The shipped default is the wrong panel for my device.** Flash this image as-is, boot it, and you get a black screen — and with no output and no obvious cause, the natural conclusion is "wrong image, download a different one," which sends you back to guessing. The fix is a file copy, but only if you know that's what's wrong.

The image does include a `SELECT MODEL.bat` → `select_device.ps1` helper for exactly this. It's Windows PowerShell. On a Mac it's two `cp` commands.

## Step 7: flashing

A fresh 128 GB card, so the stock card survives untouched as a recovery path — better than any backup.

**Do not format it first.** Writing a raw image replaces the partition table and every partition; whatever you format gets overwritten seconds later.

Identify the target *every single time*, because identifiers get recycled — my stock card was `disk4`, the new one came up as `disk5`:

```bash
diskutil list external physical
```

```
/dev/disk5 (external, physical):
   0:     FDisk_partition_scheme    *128.0 GB   disk5
   1:               Windows_NTFS     128.0 GB   disk5s1
```

Then:

```bash
diskutil unmountDisk /dev/disk5
sudo dd if=/Users/me/Downloads/dArkOSRE_R36_trixie_03082026.img of=/dev/rdisk5 bs=4m status=progress
sync
```

Two gotchas, both of which bit me:

- **`rdisk5`, not `disk5`.** The raw device bypasses the buffer cache and is roughly an order of magnitude faster.
- **zsh does not expand `~` after `=`.** `if=~/Downloads/foo.img` passes the literal string `~/Downloads/foo.img` and `dd` reports `No such file or directory` for a file that is plainly right there. That's `magicequalsubst`, off by default in zsh. Use `$HOME` or an absolute path.

Afterwards:

```
1:  Windows_FAT_32 BOOT        104.9 MB
2:  Linux                      7.9 GB
3:  Windows_FAT_32 EASYROMS    314.6 MB
    (free space)               119.7 GB
```

## Step 8: the swap

`boot.ini` loads `rk3326-r36s-linux.dtb` from the root of the BOOT partition, so overwriting in place is the entire change — no config edit needed.

**Two files, not one.** `rg351v-uboot.dtb` also differs per variant (U-Boot uses it for the early splash before the kernel takes over), and I verified the checksums don't match between variants. Copying only the kernel dtb is a great way to get a garbled boot logo and then a working desktop, which is a confusing place to end up.

```bash
SRC="/Volumes/BOOT/dtb/clone/R36S-V12 2023-08-18 Variant 3 Panel 3"
cp /Volumes/BOOT/rk3326-r36s-linux.dtb /Volumes/BOOT/rk3326-r36s-linux.dtb.orig   # reversible
cp /Volumes/BOOT/rg351v-uboot.dtb      /Volumes/BOOT/rg351v-uboot.dtb.orig
cp "$SRC/rk3326-r36s-linux.dtb" /Volumes/BOOT/
cp "$SRC/rg351v-uboot.dtb"      /Volumes/BOOT/
sync
```

Verify before ejecting — re-run the fingerprint and confirm the card now matches the stock card:

```
rk3326-r36s-linux.dtb   root=rockchip,rk3326-r36s-linux        [('sitronix,st7703', 'ee05de3b4a3a54ca', 753)]
rf351g3ka.dtb (stock)   root=rockchip,rk3326-evb-lp3-v12-linux [('sitronix,st7703', 'ee05de3b4a3a54ca', 753)]
```

Same panel. Eject properly and boot.

## It worked

First boot came up on a blue `expandtoexfat.sh` screen — growing the rootfs and building the exFAT ROM partition across the remaining ~120 GB.

Which is the answer, incidentally. **A legible screen with readable text and a rendering progress bar means the panel initialized correctly.** That was the one genuinely uncertain call in the entire process, and it got settled the moment anything drew on the display.


## Actually putting games on it

Getting the OS booting is only half the job. The content side is simple once you've seen it, and impossible to guess beforehand, so here it is.

### One folder per system

The third partition, `EASYROMS`, ships with about 150 pre-created empty folders — one per supported system:

```
3do  advision  amiga  arcade  atari2600  atarilynx  c64  cps1  daphne  doom
dos  dreamcast  famicom  gb  gba  gbc  genesis  mame  n64  neogeo  nes  ngp
psp  pspminis  psx  saturn  scummvm  sega32x  segacd  snes  ...  zxspectrum
bios  themes  tools  launchimages
```

**The folder name is the entire interface.** There's no registration step, no config file to edit, no scraper you must run first. Put a file in the folder that matches its system and EmulationStation finds it. For PSP:

```
EASYROMS/psp/        <- .iso, .cso, .pbp
EASYROMS/pspminis/   <- PSP Minis titles
```

Files go directly in, not in per-game subfolders. PSP needs no BIOS, which is worth knowing because several other systems do — PS1 and Saturn among them — and those expect their firmware dumps in the top-level `bios/` folder. A system that silently shows no games is usually a naming problem; one that shows games but refuses to launch them is usually a missing BIOS.

### Why the first boot bothered converting to exFAT

That blue `expandtoexfat.sh` screen from the previous section isn't only about reclaiming the unallocated space. It's also a format change, and the format is the point.

FAT32 cannot store a file larger than **4 GB**. Disc-based systems blow through that constantly — Dreamcast, Saturn, PS1 collections, and plenty of PSP ISOs. exFAT removes the ceiling, and macOS reads *and writes* it natively.

Which means the card behaves completely differently from the stock card back in Step 1. No "disk is not readable" dialog, no hidden partition types, no `sudo mount -t msdos`. It just mounts in Finder. Everything I did with `diskutil` earlier was a consequence of the stock firmware's odd partition layout, not something you have to live with afterwards.

### Copying

Card reader, drag into the matching folder, then eject properly:

```bash
diskutil eject /dev/diskN
```

Pulling an exFAT card mid-write is an excellent way to corrupt the directory table, and unlike the read-only mounts earlier in this post there's no safety net here — you're writing.

Network transfer is not really available: the R36S has no built-in WiFi, so SSH or Samba means a USB dongle first. For most people the card reader is less hassle.

Then **restart EmulationStation** — `Start` → `Quit` → `Restart EmulationStation`, or just reboot. The gamelist is scanned once at startup, so new files are invisible until it restarts. This accounts for approximately all "I copied the game and it's not showing up" confusion.

### What this hardware can actually run

Worth calibrating before you go to any effort. The RK3326 is four Cortex-A35 cores at roughly 1.3 GHz with a Mali-G31 — a genuinely modest part, chosen for battery life and price.

- **Excellent**: everything 2D and 8/16-bit — NES, SNES, Genesis, GBA, Neo Geo, arcade, PS1
- **Workable**: N64 and Dreamcast, per-title, with settings tweaking
- **Marginal**: PSP — lighter 2D titles often play fine, most 3D ones drop frames badly
- **No**: anything the PSP itself struggled with

PSP is the most demanding system this device claims to support, and "supported" means PPSSPP launches, not that a given game is playable. Frameskip, 1× rendering resolution, and disabling slow effects are the knobs, but they can't turn an A35 into something it isn't. Set expectations accordingly and the device is a delight at everything below that line.



## Root cause analysis: the sticks that almost worked

The screen was right, the buttons were right, and games booted. Then I launched a game that actually needs analog sticks, and it took longer to fix than everything above combined.

### Symptoms

In **Ape Escape**, the one PS1 game that *requires* dual analog sticks:

- The d-pad moved the camera. That's by design and not a bug, but it made the stick problem easy to confuse with a mapping problem.
- Pushing the left stick all the way made Spike **animate in place without moving**.
- With RetroArch's **Analog Sensitivity at 5.0**, he **walked on his own** with the stick released.
- With sensitivity 5.0 and **Analog Deadzone at 0.2**, he **didn't respond to the stick at all**.

(Ridge Racer's steering also failed, but that's a red herring. The original Ridge Racer predates analog controllers, so sticks were never going to do anything in it.)

### The wrong turn

I spent the first several attempts treating it as tuning: analog sensitivity, deadzone, Autoconfig, controller device type, core options. That's the standard advice, and it's the standard advice because cheap sticks *usually* just need a bigger deadzone.

The last symptom is what ruled that out. **A deadzone of 0.2 ignores anything under 20% of the stick's travel.** If pushing the stick all the way can't get past that, the stick is reporting less than a fifth of its range at full push. The other two symptoms fit the same explanation:

| Setting | What happened | Why |
|---|---|---|
| Defaults | Animates, doesn't move | A full push reads as a small push |
| Sensitivity 5.0 | Walks on its own | The resting offset got amplified into a push |
| + Deadzone 0.2 | No response | The entire weak range falls inside the deadzone |

No RetroArch setting can fix a signal that's weak before RetroArch receives it. **The problem was below the emulator**, at the kernel driver, which is set up by the device tree.

### The known-good reference: where `rf351g3ka.dtb` came from

The investigation depends on one file: the device tree the **factory firmware** booted on this exact unit. Steps 1–3 cover the short version. Because the whole RCA rests on this file, here's exactly how I got it and why it can be trusted.

**1. Find the boot partition on the stock card.** The original SD card that shipped with the device, not the new one, showed up as `disk4`:

```
$ diskutil list
/dev/disk4 (external, physical):
   #:                       TYPE NAME            SIZE       IDENTIFIER
   0:     FDisk_partition_scheme                *15.6 GB    disk4
   1:          DOS_FAT_16_Hidden                 1.1 GB     disk4s1
   2:                      Linux                 536.9 MB   disk4s2
   3:                      Linux                 3.0 GB     disk4s3
   4:                       0x1C                 11.0 GB    disk4s4
```

`disk4s1` is the boot partition, where bootloaders expect kernels and device trees. It's plain FAT16, but its MBR type byte is `0x16`, "hidden FAT16". Hidden partition types are an old trick to keep operating systems from auto-mounting a partition, which is why macOS offered to eject a disk it called unreadable.

**2. Get it mounted, read-only.** The obvious route failed:

```
$ diskutil mount readOnly /dev/disk4s1
Volume on disk4s1 failed to mount

$ diskutil info /dev/disk4s1
   File System:               None
   Partition Type:            DOS_FAT_16_Hidden
```

`diskutil` chooses the filesystem driver from the partition type, and it has no mapping for hidden types. Reading the raw device to check the boot sector needs root (`dd: /dev/rdisk4s1: Permission denied`). Calling the FAT driver directly skips type detection entirely:

```bash
sudo mkdir -p /Volumes/r36boot
sudo mount -t msdos -o rdonly /dev/disk4s1 /Volumes/r36boot
```

macOS loaded `msdosfs.kext` and the partition mounted. `-o rdonly` matters here: this card held the only copy of a configuration known to work on this hardware.

One mistake to avoid: the card was later pulled while this manual mount was still active, which left a zombie mount pointing at a device that no longer existed. `diskutil unmount` can't clear it. It takes `sudo umount -f /Volumes/r36boot`. Unmount before removing the card.

**3. Follow the boot chain to the file that's actually loaded.** Many clone firmware images ship several device trees and only load one, so a `.dtb` in the directory doesn't prove anything. `boot.ini` is an ODROID-GO Advance U-Boot script that picks a config by hardware revision:

```
if test ${hwrev} = 'v11'; then
sysboot mmc 1:1 any ${scriptaddr} /extlinux/rk3326-odroidgo2-linux-v11.dtb.conf
elif test ${hwrev} = 'v10-go3'; then
sysboot mmc 1:1 any ${scriptaddr} /extlinux/rk3326-odroidgo3-linux.dtb.conf
else
sysboot mmc 1:1 any ${scriptaddr} /extlinux/rk3326-odroidgo2-linux.dtb.conf
fi
```

**None of those three `.conf` files exist on the card.** The `extlinux/` directory contains only `extlinux.conf` and `rf351g3ka.dtb.conf`, so every branch fails and U-Boot falls back to the standard `extlinux/extlinux.conf`:

```
LABEL EmuELEC
  LINUX /KERNEL
  FDT /rf351g3ka.dtb
  APPEND boot=UUID=0805-2944 ... fbcon=rotate:3 ...
```

`FDT /rf351g3ka.dtb` settles it. The only `.dtb` in the partition root is the one the kernel receives, and `rf351g3ka.dtb.conf` points at the same file. There's also a runtime check: the stock firmware's display worked with `fbcon=rotate:3`, and this device tree's panel node is what drives that display.

**4. Copy it off, with context.** Everything that describes the boot went into `~/r36s-stock-boot-backup/`:

```
boot.ini
extlinux/extlinux.conf
extlinux/rf351g3ka.dtb.conf
rf351g3ka.dtb           <- the reference
rf351g3ka.dtb.txt       <- full human-readable decode
KERNEL.md5  SYSTEM.md5  fs-resize.log  get_kernel_log.sh
```

The config files came along so the device tree can be matched to the boot that used it. The stock card itself was never written to, so it's still a complete fallback. Every stick-wiring comparison below uses the copy.

**5. Read it without `dtc`.** macOS doesn't ship the device tree compiler, and the format is simple enough to parse directly:

- A **40-byte header** of ten big-endian `u32`s: magic `0xd00dfeed`, total size, offsets to the structure and strings blocks, version, and block sizes.
- A **structure block** as a token stream: `BEGIN_NODE` (1) followed by a NUL-terminated name, `PROP` (3) followed by `<length, name offset>` and the value, `END_NODE` (2), `NOP` (4), `END` (9). Everything is padded to 4-byte boundaries.
- A **strings block** holding property names, referenced by offset.

About forty lines of Python turn that into `{node path: {property: bytes}}`. That was enough for the panel fingerprint. The joystick work needed three more features:

- **Phandle resolution.** Build a `phandle → node path` map from every node's `phandle` property, because references are file-local numbers.
- **Reference decoding.** GPIO references are `<phandle pin flags>` triples (flag `1` = active-low), `io-channels` are `<phandle index>` pairs, and pinctrl groups are `rockchip,pins` quads `<bank pin function config>`.
- **GPIO bank from node name.** `gpio2@ff260000` is bank 2, so `<195 12 1>` becomes `gpio2[12], active-low` whatever `195` happens to be in that file.

Decoded that way, `/play_joystick` in the stock tree gives the "rocker" pins in the table below, including the one the dArkOS tree got wrong.

### Investigation

Back to the method from Step 4: compare against the one configuration known to work on this hardware, the stock `rf351g3ka.dtb` described above.

**The two firmwares use different joystick drivers.** Stock uses a vendor `play_joystick` driver. dArkOS uses `odroidgo3-joypad`, which reads all four stick axes through **one** ADC channel and a 4-way analog multiplexer. Two GPIO pins (`amux-a`, `amux-b`) pick which axis is connected, and an enable pin (`amux-en`) turns the chip on.

**Raw phandles lie.** GPIO references in a device tree are `<phandle pin flags>`, and phandle numbers are local to each file. `195` means one thing in the stock tree and another in dArkOS's. Comparing raw cells across files is meaningless, so I extended the parser to resolve every phandle to the node it points at:

```python
ph = {cells(props['phandle'])[0]: path for path, props in nodes.items() if 'phandle' in props}
# <195 16 1>  ->  gpio2@ff260000[16, active-low]
```

With names resolved, the comparison was straightforward:

| Role | Stock `play_joystick` | dArkOS `odroidgo3-joypad` |
|---|---|---|
| Mux select | `rocker1-gpios` = **gpio2[16]** | `amux-a-gpios` = **gpio2[16]** ✅ |
| Mux select | `rocker-gpios` = **gpio2[15]** | `amux-b-gpios` = **gpio2[15]** ✅ |
| Mux enable | `rocker0-gpios` = **gpio2[12]** | `amux-en-gpios` = **gpio2[11]** ❌ |
| Buttons (18) | gpio3[10–25], gpio2[13–14] | same pins ✅ |

The vendor calls them "rocker" pins, but these are the same multiplexer control lines. Two of the three agree. **The enable line doesn't.** dArkOS drives gpio2[11], which the stock firmware never touches. With the chip never enabled, the ADC reads a floating input, which produces exactly the weak, drifting signal from the table above. The button pins all match, which is why the buttons worked fine.

**Corroboration from the repo.** A pin scan across all 27 device trees in the image found that `clone/R36S-V12 2023-08-18 Variant 2 Panel 1` is the only clone variant using enable = gpio2[12]. Its joystick node is **identical** to mine apart from that pin, file-local phandle numbers, and two swapped labels. It isn't a usable replacement, though. Its panel blob is `3f7a1a32…`, not mine, and it differs from my device tree in 230 other properties (DDR timing, eMMC, battery). So the right joystick wiring *existed* in the repo, just never combined with my panel.

**Conflict check.** Before driving a new pin, I checked what else claims gpio2[12]. The pinctrl groups mentioning it are `uart2-m1` (selected by no node) and the camera interface's `dvp-d0d1-m0` (selected only by `cif` and `rk_isp`, both `status = "disabled"`). So nothing active uses it.

### Fix

A one-byte patch to the device tree I already had, leaving the panel config alone:

```python
# find the amux-en-gpios property inside /odroidgo3-joypad, then:
before = struct.unpack('>3I', data[off:off+12])   # (195, 11, 1)
assert before[1] == 11
data[off+4:off+8] = struct.pack('>I', 12)          # (195, 12, 1)
```

```
amux-en-gpios at file offset 0xf0e0: (195, 11, 1) -> (195, 12, 1)
bytes changed: 1 at ['0xf0e7']
md5 original: 7a43ef8c4a9336ff4cdbb25add874e12
md5 patched : d4f7c8872302f4720ecf6ba688dc425a
```

Since the value is the same size, patching in place is safe. Nothing moves, no offsets shift, and the file doesn't need to be recompiled. Before ejecting I re-parsed it and confirmed the enable pin reads 12, the panel blob still hashes to `ee05de3b…`, and the original is kept on the card as `rk3326-r36s-linux.dtb.v3p3-pin11` for a one-`cp` rollback.

After booting, with sensitivity back at 1.0 and a small deadzone, Spike stood still when I let go and ran when I pushed. Fixed.

### Second fix: the inverted right stick

With the multiplexer enabled, the sticks finally reported their full range, and that exposed a second bug the first one had been hiding. **The right stick's axes were both inverted**: up read as down and left as right. In most games that's annoying. In Ape Escape it breaks the game, because gadget attacks go in the right stick's direction.

**Cause.** The `odroidgo3-joypad` driver reads each axis as a raw ADC value, and some boards wire the potentiometers backwards. The upstream driver ([hardkernel `odroidgo3-joypad.c`](https://github.com/hardkernel/linux/blob/odroidgoA-4.4.y/drivers/input/joystick/odroidgo3-joypad.c)) handles this with a per-channel `invert` flag that reports `max - value` instead of `value`. In the device tree, the flags are empty boolean properties. Mine had:

```
/odroidgo3-joypad : invert-absx    (flag)
/odroidgo3-joypad : invert-absy    (flag)
```

Those are the **left** stick's axes (`ABS_X`, `ABS_Y`). There was nothing for the right stick's `ABS_RX` and `ABS_RY`, so on this board it came through mirrored on both axes.

**Does the running kernel support the right-stick flags?** The repo couldn't answer that. A scan of all 27 device trees in the image found only `invert-absx` and `invert-absy`, in every variant. The flags' names (`invert-absrx`/`invert-absry`) come from a vendor fork, not upstream, and an unrecognized device tree property does nothing and logs nothing. So I checked the kernel itself. Property names a driver looks up are stored as plain text in the uncompressed arm64 `Image` on the BOOT partition:

```
$ strings -n 6 /Volumes/BOOT/Image | grep -E '^invert-abs'
invert-absrx
invert-absry
invert-absx
invert-absy

$ strings -n 6 /Volumes/BOOT/Image | grep -m1 '^Linux version'
Linux version 4.4.189 (lcdyk@Lcdyk) (gcc version 6.3.1 20170404 (Linaro GCC 6.3-2017.05) ) #62 SMP Sat Dec 13 08:49:45 CST 2025
```

Both right-stick names are compiled in next to the left-stick ones, so the running kernel's driver looks for them. The same `strings` output also contained `amux-en-gpios not configured; leaving MUX EN unchanged`, which fits the first bug: the driver never checks whether the enable pin it was given actually does anything.

**This patch changes the file size.** The pin fix swapped one 4-byte value for another. Adding two properties inserts new tokens into the structure block and new names into the strings block, which moves every offset after them. Without `dtc`, that meant writing a device tree *serializer*, and it had to be trustworthy before it touched anything.

The test: parse a device tree, write it back out unmodified, and require byte-for-byte identical output. The first attempt failed:

```
round-trip size   : 92358 -> 92549
round-trip identical: False
first diffs at ['0x6', '0x7', '0x22', '0x23', '0x466', '0x467', ...] count 106
```

191 bytes too big, with differences in the header size fields and throughout the structure block. The cause is a `dtc` space-saving trick: it **merges property names by suffix**, so a name can be stored as the tail of a longer name that ends the same way, pointing into the middle of it. My writer only reused a name when it found it as a whole string, so shared names were written out again as duplicates. The fix was to record each property's original name offset when parsing and write that same offset back. Only genuinely new names get appended. After that:

```
rk3326-r36s-linux.dtb        size 92358 -> 92358   identical=True
rf351g3ka.dtb                size 107933 -> 107933 identical=True
```

It reproduced both the patched dArkOS tree and the stock tree exactly, so any difference in its output would come only from an intentional change.

**The patch.** I added two empty properties right after `invert-absy`, starting from the pin-12 tree:

```python
root, rsv, strings, hdr = load(data)
joy = find(root, b'odroidgo3-joypad')
i = [p[0] for p in joy['props']].index(b'invert-absy') + 1
joy['props'][i:i] = [[b'invert-absrx', b''], [b'invert-absry', b'']]
out = dump(root, rsv, strings, hdr)
```

I verified it by re-parsing both files and comparing every property of every node, not just the joystick node:

```
size        : 92358 -> 92408
added       : [('//odroidgo3-joypad', 'invert-absrx'), ('//odroidgo3-joypad', 'invert-absry')]
removed     : []
changed     : []
amux-en pin still       : 12
panel blob              : ee05de3b4a3a54ca
md5 v2                  : f9a095c1db3c06506c70900d025219a9
```

The +50 bytes check out: two property headers at 12 bytes each, plus `invert-absrx\0` and `invert-absry\0` appended to the strings block at 13 bytes each.

**Install.** Same routine as before. Before copying anything, the script refused to continue unless the card's current device tree hashed to the pin-12 version (`d4f7c887…`). It kept that file on the card as `rk3326-r36s-linux.dtb.v1-pin12` for rollback, copied v2 into place, re-checked the hash, flags, pin and panel on the card itself, and ejected. The on-device workaround (rebinding the right stick's directions in RetroArch) was an option, but it would only fix RetroArch and would need undoing once the real fix went in.

After booting, the right stick aimed gadgets in the direction pushed. Both sticks work correctly.

### Why the earlier process missed it

- **The panel fingerprint only checks the panel.** Hashing `panel-init-sequence` narrowed 44 device trees to 6 and was right about the screen. It said nothing about the rest of the board, and I picked among the six using the `compatible` string and community test status.
- **All six panel matches shared the wrong enable pin.** Any of them would have produced this bug. The one variant with the correct pin had the wrong panel, so no device tree in the repo matched this board on both.
- **Buttons working felt like confirmation.** I'd said "if the buttons misbehave, it's the wrong board." They didn't, because button pins and stick-mux pins are separate lines. That test was too weak.
- **The first bug hid the second.** With the multiplexer never enabled, the sticks barely registered, so an inverted axis was impossible to notice. Inversion only becomes visible once the signal is good.
- **Cheap sticks make it easy to blame the hardware.** Drift and weak response are also what worn-out potentiometers do, so "the sticks are just bad" was a plausible, wrong explanation.

### Contributing factors

- The motherboard silkscreen was never read. It wouldn't have fixed anything by itself, since the pin was wrong in every matching variant, but it would have told me whether this board is really a "Variant 3" or something no variant covers.
- The upstream `odroidgo3-joypad` binding gives no error for a mux that never enables. A floating ADC just reads noise, so the failure is silent.

### Follow-up

This deserves an upstream report to [southoz/dArkOSRE-R36](https://github.com/southoz/dArkOSRE-R36): board strings, the panel and pin diffs, and both patches (`amux-en-gpios` → gpio2[12], plus `invert-absrx`/`invert-absry`). As far as I can tell no variant in the repo gets this board's joystick wiring right, so anyone else with this panel and this board gets the same almost-working sticks.


## What I'd take away from this

- **The device knows what it is.** Every clone handheld carries a working configuration on its stock card. Reading it is faster and more reliable than matching photos of motherboards against forum posts.
- **Hash the opaque blob, not the label.** The `compatible` string was wrong 5 times out of 6 for my panel. The init sequence was right every time. Any time a config format has a human-assigned name *and* a machine-generated payload, trust the payload.
- **Verify the thing you're about to install, not just the thing you have.** Mounting the image read-only before flashing is what surfaced that the shipped default was the wrong panel — a failure that otherwise presents as an unexplained black screen.
- **Read-only until proven otherwise.** `-o rdonly`, `hdiutil -readonly`, `.orig` backups. The stock card is irreplaceable; the 8.3 GB image is a re-download.
- **A matching fingerprint only vouches for what it hashes.** The panel hash was right and said nothing about joystick wiring. Check each subsystem you care about against the known-good config, not just the one that was easiest to fingerprint.

The loose end turned out to matter. The panel was confirmed empirically the moment the screen lit up. The *board* half was an inference, and "the buttons work" was too weak a test for it. The sticks were wired to a pin no candidate device tree used, and once that was fixed, the right stick turned out to be inverted too. Both are covered in the root cause analysis above. The fixes were one byte and two empty properties.

The motherboard silkscreen is still unread, under four screws. Reading the stock device tree turned out to be more useful than opening the case would have been.
